# Extract training samples
* Download low-tide cloud free satellite iamges closest to the UAV image collection
* Sample the satellite image bands where appromximately a single UAV class

In [1]:
import pathlib
import numpy
import dask.distributed

module_path = pathlib.Path.cwd().parent / 'scripts'
import sys
if str(module_path) not in sys.path:
    sys.path.append(str(module_path))
import utils
import sentinel2
import training
import sampling

%load_ext autoreload
%autoreload 2

# Values to edit

In [2]:
sample_method = "sampling_2" # sampling_1 sampling_2
method_2_threshold = .8 # .95 .97 .98 .99 1.0
max_cloud_cover = 1 # percentage
low_tide_delta = 1 # hours

site_names = ["CatlinsLake", "CatlinsRiverMouth", "Childrens", "Duvauchelle", "Robinsons", "Takamatua", "Purau", "Ihutai",
              "IveyBay_Nov25", "IveyBay_Feb26", "LeftBank_Nov25", "LeftBank_Feb26", "Paremata_Nov25", "Paremata_Feb26",
              "Paremata_Feb25", "ThePoint_Nov25", "ThePoint_Feb26", "Takapuwahia_Nov25", "Takapuwahia_Feb26", "IveyBay_ThePoint_LeftBank_Oct24"]

# Cells to run

In [3]:
cluster = dask.distributed.LocalCluster()
client = dask.distributed.Client(cluster)
display(client)

Connection method: Cluster object,Cluster type: distributed.LocalCluster
Dashboard: http://127.0.0.1:8787/status,
Dashboard: http://127.0.0.1:8787/status,Workers: 4
Total threads: 8,Total memory: 31.73 GiB
Status: running,Using processes: True
Comm: tcp://127.0.0.1:59204,Workers: 4
Dashboard: http://127.0.0.1:8787/status,Total threads: 8
Started: Just now,Total memory: 31.73 GiB
Comm: tcp://127.0.0.1:59223,Total threads: 2
Dashboard: http://127.0.0.1:59224/status,Memory: 7.93 GiB
Nanny: tcp://127.0.0.1:59207,


2026-09-08 09:11:04,432 - distributed.scheduler - WARNING - Worker failed to heartbeat for 61464s; attempting restart: <WorkerState 'tcp://127.0.0.1:59223', name: 0, status: running, memory: 0, processing: 0>
2026-09-08 09:11:07,476 - distributed.nanny - WARNING - Restarting worker


In [4]:
data_path = utils.get_data_path()
utils.create_data_folders()

training_labels_file = data_path / "ELF24505_ClassificationClasses.txt"
lowtide_search_range_file = data_path / "ELF24505_satellite_day_search_range.csv"
servey_dates_file = data_path / "ELF24505_SurveyDates.csv" 
uav_folder = data_path / "classified_uav"
samples_folder = utils.get_samples_path(sample_method=sample_method, method_2_threshold=method_2_threshold,
                                        max_cloud_cover=max_cloud_cover, low_tide_delta=low_tide_delta)

In [13]:
p=pathlib.Path(r"C:\Local\repos\seagrass-detection\data\training\low_tide_delta_1_max_cloud_percentage_1\sampling_2_80_percent\CatlinsLake_training_data.csv")

In [20]:
list(p.parent.iterdir())

[]

In [18]:
p.exists()

False

In [22]:
for site_name in site_names:
    sampling.sample_site(
        site_name=site_name,
        training_labels_file=training_labels_file,
        uav_folder=uav_folder,
        sample_method=sample_method,
        method_2_threshold=method_2_threshold,
        max_cloud_cover=max_cloud_cover,
        low_tide_delta=low_tide_delta
    )
counts_summary = sampling.site_sample_counts_by_class(sample_method=sample_method, method_2_threshold=method_2_threshold,
                                                      max_cloud_cover=max_cloud_cover, low_tide_delta=low_tide_delta)

Site CatlinsLake
Site CatlinsRiverMouth
Site Childrens
Site Duvauchelle
Site Robinsons
Site Takamatua
Site Purau
Site Ihutai
Site IveyBay_Nov25
Site IveyBay_Feb26
Site LeftBank_Nov25
Site LeftBank_Feb26
Site Paremata_Nov25
Site Paremata_Feb26
Site Paremata_Feb25
Site ThePoint_Nov25
Site ThePoint_Feb26
Site Takapuwahia_Nov25
Site Takapuwahia_Feb26
Site IveyBay_ThePoint_LeftBank_Oct24


In [26]:
counts_summary[['Seagrass', 'Seagrass submerged', 'Gracilaria', 'Gracilaria submerged', 'Ulva', 'Ulva mats', 'Unvegetated',
'Water', 'Terrestrial', 'Submerged vegetation', 'Microphytobenthos', 'Rock', 'Saltmarsh' , 'Shadow', 'Glare']].astype(int) 

uav_class_name,Seagrass,Seagrass submerged,Gracilaria,Gracilaria submerged,Ulva,Ulva mats,Unvegetated,Water,Terrestrial,Submerged vegetation,Microphytobenthos,Rock,Saltmarsh,Shadow,Glare
Site,,,,,,,,,,,,,,,
CatlinsLake,0,0,1532,1533,0,0,25908,7112,0,0,0,0,0,0,0
CatlinsRiverMouth,748,0,0,0,0,0,3958,358,1658,0,0,180,0,0,84
Childrens,381,18,0,0,0,0,1476,2169,0,12,54,0,0,0,0
Duvauchelle,2240,560,0,0,0,0,2144,5436,0,0,60,0,0,14,0
Ihutai,4474,0,4806,0,842,3380,70266,16574,716,6938,0,0,3146,0,0
IveyBay_Feb26,17,0,0,0,0,0,46,0,1,0,0,0,0,0,0
IveyBay_Nov25,6,0,0,0,0,0,158,6,2,0,0,0,0,0,0
IveyBay_ThePoint_LeftBank_Oct24,10,0,0,0,0,0,188,410,0,46,0,0,0,12,0
LeftBank_Feb26,54,0,0,0,0,0,175,92,0,0,0,0,0,0,0


In [24]:
print(f"Samples located at {samples_folder}")

Samples located at C:\Local\repos\seagrass-detection\data\training\low_tide_delta_1_max_cloud_percentage_1\sampling_2_80_percent
